# Prompt Engineering

**Live online course — instructor walkthrough notebook**

This notebook follows the course lecture notes. Each section has:
- **Lecture notes** (markdown) — what to explain on the slide/screen.
- **Demo code** — cells to run live for students to see real output.
- **🏫 During class** callouts — specific instructor actions, questions to ask, and variations to try.

The running example throughout the module is a **one-day meal planner for athletes**. We start with a weak prompt, apply one technique at a time, and re-evaluate so students can watch the score climb in real time.

---

## Agenda

1. Prompt Engineering — the loop
2. Being Clear and Direct
3. Being Specific (attributes + steps)
4. Structure with XML Tags
5. Providing Examples (one-shot / multi-shot)
6. Recap + practice exercises

## 0. Setup (do this before class starts)

1. Install dependencies:
   ```bash
   pip install anthropic python-dotenv
   ```
2. Create a file named `.env` in the same directory as this notebook containing:
   ```
   ANTHROPIC_API_KEY="sk-ant-...your-key..."
   ```
3. Add `.env` to `.gitignore` so it is never committed to version control.

> **🏫 During class:** This notebook assumes students already sat through *Intro to Claude API* and *Prompt Evaluation*. A quick refresher: *“We measure prompts with an eval so improvements are visible; today we go the other direction — we write the prompt and watch the score move.”*

In [ ]:
# Install packages (uncomment if not already installed)
%pip install anthropic python-dotenv

The setup cell below does four things students should recognize from *Intro to Claude API*:

1. `load_dotenv()` reads the `.env` file next to the notebook so `os.getenv("ANTHROPIC_API_KEY")` returns the secret — the key never appears in the notebook.
2. `client = anthropic.Anthropic()` creates the SDK client with no arguments; it picks up `ANTHROPIC_API_KEY` from the environment automatically.
3. `model = "claude-haiku-4-5"` — we deliberately pick the **smallest** model for this module. Prompt engineering wins are most visible on a weaker model; a stronger model papers over bad prompts.
4. The three `print(...)` lines are a sanity check. If `Key loaded:` prints `False`, the `.env` file isn't where it needs to be — fix that before running any API call.

In [ ]:
from dotenv import load_dotenv
import anthropic
import os

load_dotenv()  # loads ANTHROPIC_API_KEY from .env

client = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY automatically

# Prompt engineering gains are most visible on a smaller model.
# We'll deliberately use Haiku so weak prompts look weak and strong prompts look strong.
model = "claude-haiku-4-5"

print("SDK version:", anthropic.__version__)
print("Model:", model)
print("Key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

### Shared helpers

Every demo in this notebook reuses the same `chat()` helper. After this cell runs, students should see we're only varying **one thing** per demo — the prompt itself.

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send messages to Claude and return the assistant text."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

# A fixed athlete profile we'll reuse to keep comparisons fair.
athlete = {
    "height_cm": 178,
    "weight_kg": 72,
    "goal": "build lean muscle for a half-marathon cycle",
    "dietary_restrictions": "vegetarian, no dairy",
}

---
# 1. Prompt Engineering — the loop

**Prompt engineering** is the practice of improving prompts to get more reliable, higher-quality outputs from language models. It's not "magic words" — it's a measurable, iterative loop.

### The loop (this is the whole module)

```
write initial prompt  →  interpolate inputs  →  run evaluation
         ↑                                              |
         └──  apply one technique  ←──────  read score ──┘
```

### Running example
Generate a **one-day meal plan for an athlete** using their height, weight, physical goal, and dietary restrictions.

### How we measure (recap from the eval module)
| Piece | What it does |
|---|---|
| `prompt_input_spec` | Dictionary listing the inputs the prompt expects (e.g., `height_cm`, `dietary_restrictions`). |
| `generate_dataset()` | Creates test cases — many athlete profiles — from that spec. |
| `run_prompt()` | Interpolates each test case into the prompt and sends it to Claude. |
| `extra_criteria` | Extra rules the grading model enforces ("must include calorie total", "must name specific foods", etc.). |
| `max_concurrent_tasks` | How many test cases to run in parallel — tune to your rate limit. |
| `output.html` | Rendered report: each test case, the generated meal plan, and the grader's score. |

### What to expect numerically
Starting with a naive prompt on a small model, expect a score around **2.32 / 10**. We'll climb from there, one technique at a time.

### Why this loop matters
Without an eval, "the prompt got better" is a feeling. With an eval, it's a number. Every technique in the rest of this notebook is something students should apply *and then re-run the eval*.

### Demo: the starting prompt (score ~2.32)

This cell shows what a typical "I just typed something" prompt looks like, and what Haiku does with it on a single athlete profile. **Watch:** no structure, no format, no instructions — and the response reflects that. As we add techniques in the next sections, the same profile should produce a much stronger plan.

In [ ]:
# Initial, intentionally weak prompt: just state the goal and drop the data in
initial_prompt = f"""Meal plan for an athlete.
Height: {athlete['height_cm']} cm
Weight: {athlete['weight_kg']} kg
Goal: {athlete['goal']}
Restrictions: {athlete['dietary_restrictions']}
"""

msgs = []
add_user_message(msgs, initial_prompt)
print(chat(msgs, temperature=0.0))

> **🏫 During class:**
> 1. Run the cell and leave the output visible — we will keep referring back to it.
> 2. Say out loud: *“Score on the eval: **2.32**. By the end of the notebook we're going to 8-plus with the same model and the same inputs — only the prompt changes.”*
> 3. Ask the room: *“Looking at this output, name one thing it's missing that an athlete would actually need.”* Collect 2–3 answers — they'll map directly onto the techniques that follow.

---
# 2. Being Clear and Direct

**Being clear and direct** = use simple, direct language with an **action verb** in the **first line** of the prompt to specify the exact task.

### Why the first line matters most
The first line sets the foundation for everything the model generates. If it's vague, every token afterwards is trying to guess what you actually wanted.

### Structure
```
<action verb> <direct task> <output specifications>
```

### Examples
| Vague | Clear and direct |
|---|---|
| *"solar panels"* | *"Write three paragraphs about how solar panels work."* |
| *"geothermal info"* | *"Identify three countries that use geothermal energy and for each include generation stats."* |
| *"meal plan for athlete"* | *"Generate a one-day meal plan for an athlete that meets their dietary restrictions."* |

### Expected lift
On the meal-plan eval, this single change took the score from **2.32 → 3.92**. No model change. No examples. Just a better first line.

### Demo: rewrite the first line

The only change from section 1 is the first sentence: it now starts with an action verb (*Generate*) and states exactly what we want. The interpolated athlete data is identical. **Watch:** the response should become more task-shaped — actual meals rather than generic advice.

In [ ]:
clear_prompt = f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

Height: {athlete['height_cm']} cm
Weight: {athlete['weight_kg']} kg
Goal: {athlete['goal']}
Restrictions: {athlete['dietary_restrictions']}
"""

msgs = []
add_user_message(msgs, clear_prompt)
print(chat(msgs, temperature=0.0))

> **🏫 During class:**
> 1. Diff the two prompts on screen — the only change is the first line. Everything else is identical.
> 2. Say out loud: *“That one-line change alone moves the eval from 2.32 to 3.92. This is the cheapest win in prompt engineering.”*
> 3. Ask a student to propose an even stronger first line (e.g., specifying meals, calories, timing) — try it live and note that specificity is *section 3*, coming up next.

---
# 3. Being Specific (attributes + steps)

**Being specific** = add **guidelines** that steer the output in a particular direction. There are two flavors, and they do different jobs.

| | **Type A — Attributes** | **Type B — Steps** |
|---|---|---|
| Controls | The **output's** qualities | The **model's reasoning process** |
| Example | "Include calorie totals. Format as a table. Use metric units." | "First compute TDEE. Then split macros. Then assign foods." |
| When to use | Almost **always** | Complex problems needing broader perspective |

### Combine them
Professional prompts almost always stack both: Type B guides *how* the model thinks; Type A constrains *what* it emits.

### Expected lift
On the meal-plan eval, adding guidelines jumped the score from **3.92 → 7.86**. This is usually the single biggest win in the loop.

### Demo: stack Type B (steps) + Type A (attributes)

The prompt below keeps the clear first line from section 2, then adds two blocks:
- **Reasoning steps** telling the model how to approach the task (compute TDEE → set macros → assign foods).
- **Output attributes** pinning the format (3 meals + 2 snacks, per-item grams, total calories + macros at the bottom).

**Watch:** the response becomes concrete and structured. Swap individual bullets in and out live to show each one's effect.

In [ ]:
specific_prompt = f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

Follow these steps:
1. Estimate the athlete's daily calorie target using the Mifflin-St Jeor equation and an activity multiplier appropriate for their goal.
2. Split the target into macros (protein g, carbs g, fat g) suitable for endurance + lean-muscle goals.
3. Choose specific foods that hit those macros AND respect every dietary restriction.
4. Distribute the foods across breakfast, lunch, dinner, and two snacks.

The output must:
- Use a single Markdown table with columns: Meal | Item | Portion (g) | Calories | Protein (g).
- Include breakfast, lunch, dinner, and two snacks (5 rows minimum).
- End with a one-line total for calories and macros.
- Use metric units throughout.
- Contain zero meat, fish, or dairy.

Athlete:
- Height: {athlete['height_cm']} cm
- Weight: {athlete['weight_kg']} kg
- Goal: {athlete['goal']}
- Restrictions: {athlete['dietary_restrictions']}
"""

msgs = []
add_user_message(msgs, specific_prompt)
print(chat(msgs, temperature=0.0))

> **🏫 During class:**
> 1. Run the cell. Point at the table and call out: *“Notice — metric units, no dairy, calorie totals. Every one of those is a bullet in the prompt.”*
> 2. Delete just the **"Follow these steps"** block and re-run. The table still appears (Type A holds the format) but the macro math often gets sloppier — that's what Type B was buying us.
> 3. Ask the room: *“Is there a guideline we should add for this specific athlete that isn't here yet?”* Good answers: hydration, pre-run timing, sodium. Try adding one and re-running.

---
# 4. Structure with XML Tags

**XML tags** organize and delineate different content sections inside a prompt so the model can tell input apart from instructions, one document from another, and examples from the task.

### Why they help
When you interpolate large amounts of content into a prompt — a document, a user record, a stack of examples — plain text runs together. Tags give the model explicit boundaries.

### Pick descriptive names
| Weak | Strong |
|---|---|
| `<data>` | `<sales_records>` |
| `<text>` | `<athlete_information>` |
| `<code>` | `<my_code>` / `<docs>` |

A debugging prompt with code **and** documentation becomes clearer when the two are in `<my_code>` and `<docs>` blocks. Same content, obvious structure.

### When to wrap
- Any interpolated input, **even if it's short**. `<athlete_information>...</athlete_information>` is worth doing for four lines of data.
- Any time you have more than one "thing" in a prompt — input + instructions, task + examples, two documents, etc.

### Demo: wrap the athlete data in a descriptive tag

The prompt below is the same as section 3, with one surgical change: the four athlete fields are now inside `<athlete_information>...</athlete_information>`. **Watch:** small content but the model stops "mixing" the athlete data with the instructions — that matters more as the input gets longer (imagine pasting a whole medical intake form in there).

In [ ]:
xml_prompt = f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

Follow these steps:
1. Estimate the athlete's daily calorie target using the Mifflin-St Jeor equation and an activity multiplier appropriate for their goal.
2. Split the target into macros (protein g, carbs g, fat g) suitable for endurance + lean-muscle goals.
3. Choose specific foods that hit those macros AND respect every dietary restriction.
4. Distribute the foods across breakfast, lunch, dinner, and two snacks.

The output must:
- Use a single Markdown table with columns: Meal | Item | Portion (g) | Calories | Protein (g).
- Include breakfast, lunch, dinner, and two snacks (5 rows minimum).
- End with a one-line total for calories and macros.
- Use metric units throughout.
- Contain zero meat, fish, or dairy.

<athlete_information>
Height: {athlete['height_cm']} cm
Weight: {athlete['weight_kg']} kg
Goal: {athlete['goal']}
Restrictions: {athlete['dietary_restrictions']}
</athlete_information>
"""

msgs = []
add_user_message(msgs, xml_prompt)
print(chat(msgs, temperature=0.0))

> **🏫 During class:**
> 1. Show the diff from section 3 — the only change is four lines wrapped in a tag. Emphasize: *“You do this even for four lines. Habit now, payoff later.”*
> 2. Variation to run live: rename the tag to a useless one like `<data>` and re-run. Often the model's referencing gets vaguer ("based on the data provided…"). Then rename it to `<athlete_information>` — the response tends to reference the athlete by attribute.
> 3. Ask: *“Where would you put the instructions if you had both the athlete's data **and** a list of disallowed foods pulled from an allergy database?”* Pull that out into two tags live.

---
# 5. Providing Examples (one-shot / multi-shot)

**One-shot / multi-shot prompting** = include one (or several) worked examples in the prompt so the model can pattern-match on both the format and the quality bar.

### When it helps most
| Use case | Why examples help |
|---|---|
| Corner cases (sarcasm, edge inputs) | Instructions can't enumerate every case; examples show the judgment call. |
| Complex output formats (JSON, nested structures) | An example is worth a thousand "make sure to include…" bullets. |
| Specific style / tone | Copying style from prose is easier than defining it. |

### How to structure an example
- Wrap it in XML tags so it's obviously a reference — not the current task.
- Include **both** the sample input and the ideal output.
- Add a short note explaining **why** the output is ideal — this reinforces the qualities you want copied.
- Add explicit guidance for corner cases (*"be especially careful with sarcasm"*, *"never assume the athlete eats eggs unless stated"*).

### Where to put examples
**After** the instructions and output spec, **before** the actual task input. Order: instructions → examples → task.

### Pro tip
Pull your highest-scoring outputs from the prompt-evaluation module and reuse them here as examples. The eval has already told you which outputs are best — that's your example library.

### Demo: one-shot meal plan with a reasoned example

The prompt below layers all four prior techniques (clear first line, guidelines, XML tags) and then adds a single worked example: a different athlete profile with an ideal meal plan *and* a short reason block explaining what makes that plan good. **Watch:** the new response usually mirrors the example's structure (header → table → totals line) without us having to specify those details in prose.

In [ ]:
one_shot_prompt = f"""Generate a one-day meal plan for an athlete that meets their dietary restrictions.

Follow these steps:
1. Estimate the athlete's daily calorie target using the Mifflin-St Jeor equation and an activity multiplier appropriate for their goal.
2. Split the target into macros (protein g, carbs g, fat g) suitable for endurance + lean-muscle goals.
3. Choose specific foods that hit those macros AND respect every dietary restriction.
4. Distribute the foods across breakfast, lunch, dinner, and two snacks.

The output must:
- Use a single Markdown table with columns: Meal | Item | Portion (g) | Calories | Protein (g).
- Include breakfast, lunch, dinner, and two snacks (5 rows minimum).
- End with a one-line total for calories and macros.
- Use metric units throughout.
- Respect every dietary restriction exactly — do not substitute “close” foods.

<example>
<athlete_information>
Height: 165 cm
Weight: 58 kg
Goal: Olympic-distance triathlon prep
Restrictions: gluten-free, no shellfish
</athlete_information>
<ideal_output>
**Daily target: 2450 kcal — 130 g protein / 320 g carbs / 70 g fat**

| Meal      | Item                          | Portion (g) | Calories | Protein (g) |
|-----------|-------------------------------|-------------|----------|-------------|
| Breakfast | Rolled oats (GF) + banana     | 80 + 120    | 430      | 11          |
| Snack 1   | Greek yogurt + honey          | 200 + 20    | 220      | 20          |
| Lunch     | Grilled chicken + rice + veg  | 150 + 200 + 150 | 680  | 42          |
| Snack 2   | Rice cakes + almond butter    | 40 + 30     | 320      | 9           |
| Dinner    | Salmon + quinoa + greens      | 170 + 180 + 150 | 800  | 48          |

**Total: 2450 kcal — 130 g protein / 318 g carbs / 72 g fat**
</ideal_output>
<why_this_is_ideal>
- Hits the calorie/macro target within 5 kcal and 5 g of each macro.
- Every item is naturally gluten-free (explicitly marks GF oats) and contains no shellfish.
- Meals scale around heavy-training days: carb-forward pre/post workout, protein spread evenly.
- Markdown table matches the required column order exactly.
</why_this_is_ideal>
</example>

<athlete_information>
Height: {athlete['height_cm']} cm
Weight: {athlete['weight_kg']} kg
Goal: {athlete['goal']}
Restrictions: {athlete['dietary_restrictions']}
</athlete_information>
"""

msgs = []
add_user_message(msgs, one_shot_prompt)
print(chat(msgs, temperature=0.0))

> **🏫 During class:**
> 1. Before running, call out the structure on screen: *instructions first, then `<example>` with both input AND ideal output AND reasoning, then the real `<athlete_information>`*. That order matters.
> 2. Run the cell. Point out how the new plan mirrors the example's header-then-table-then-totals shape — without us specifying that anywhere in prose.
> 3. Variation to try live: delete the `<why_this_is_ideal>` block and re-run. The format usually still copies, but the quality signals (macros landing within 5 g, labeling GF oats explicitly) tend to drop. That's what the reasoning was buying us.
> 4. Ask: *“Where do we get good examples from?”* — answer: the high-scoring outputs from last week's eval. Examples aren't invented; they're harvested.

---
# 6. Recap + practice exercises

### Recap (run through these out loud)
- **Prompt engineering is a loop:** write → eval → read score → apply one technique → re-run. Don't eyeball improvements; measure them.
- **Be clear and direct:** action verb in the first line. Cheapest single win (2.32 → 3.92 on our eval).
- **Be specific:** attributes (Type A) constrain the output; steps (Type B) guide the reasoning. Combine both (3.92 → 7.86).
- **XML tags:** give interpolated content explicit boundaries and descriptive names — even for short content.
- **Examples:** one-shot or multi-shot with a `<why_this_is_ideal>` block transfers both format and quality bar. Harvest them from your highest-scoring eval runs.
- **Order inside a prompt:** instructions → output spec → examples → task input.

### Exercises (do the first in class, assign the rest)
1. **Rewrite the first line:** Take your own starting prompt from the eval module and rewrite only the first line using an action verb. Re-run the eval and record the score change.
2. **Stack attributes + steps:** Add a *“Follow these steps…”* block AND an *“The output must…”* block to that prompt. Re-run the eval.
3. **Wrap your inputs:** Replace every interpolated field with a descriptive XML tag (e.g., `<athlete_information>`, `<allergy_list>`). Re-run.
4. **One-shot from your eval:** Pick your highest-scoring output from exercise 2, convert it into an `<example>` block with `<ideal_output>` and `<why_this_is_ideal>`, and add it to the prompt. Re-run.
5. **Multi-shot for a corner case:** Add a second example that covers a deliberately tricky profile (e.g., severe allergies + unusual goal). Verify the score on tricky test cases specifically improves more than the score on easy cases.

In [ ]:
# Exercise 1 scaffold — finish this live in class together.
# Uncomment to run.
#
# # Start from a weak prompt and rewrite ONLY the first line.
# weak_prompt = f"""meal plan for athlete
# Height: {athlete['height_cm']} cm
# Weight: {athlete['weight_kg']} kg
# Goal: {athlete['goal']}
# Restrictions: {athlete['dietary_restrictions']}
# """
#
# # TODO: rewrite the first line using an action verb + direct task + output specs.
# rewritten_prompt = weak_prompt
#
# msgs = []
# add_user_message(msgs, rewritten_prompt)
# print(chat(msgs, temperature=0.0))